# Задача 7.18: индикатриса Дюпена

**Задача.** Найти уравнение индикатрисы Дюпена для заданной поверхности в указанной точке.

Если $B=Ldu^2+2Mdudv+Ndv^2$ — вторая квадратичная форма, то в координатах касательной плоскости $\xi=\alpha r_u+\beta r_v$ индикатриса задается уравнением

$$|L\alpha^2+2M\alpha\beta+N\beta^2|=1.$$

In [ ]:
# Базовые библиотеки для аналитики и визуализации
import numpy as np
import matplotlib.pyplot as plt
from math import sin, cos, tan, sqrt, pi

try:
    import sympy as sp
except ImportError:
    sp = None

EPS = 1e-9
plt.rcParams['figure.figsize'] = (8, 6)
plt.rcParams['axes.grid'] = True

In [ ]:
def setup_2d(xlim=(-5, 5), ylim=(-5, 5), title=None, xlabel='x', ylabel='y'):
    """Создает 2D-плоскость с осями координат и равным масштабом."""
    fig, ax = plt.subplots()
    ax.axhline(0, linewidth=1)
    ax.axvline(0, linewidth=1)
    ax.set_xlim(*xlim)
    ax.set_ylim(*ylim)
    ax.set_aspect('equal', adjustable='box')
    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)
    if title:
        ax.set_title(title)
    return fig, ax


def plot_points_2d(ax, points, labels=None):
    """Рисует точки на 2D-графике."""
    labels = labels or [None] * len(points)
    for p, label in zip(points, labels):
        p = np.asarray(p, dtype=float)
        ax.scatter(p[0], p[1], s=45)
        if label:
            ax.text(p[0], p[1], '  ' + label)


def plot_parametric_2d(ax, xy_func, t_range, n=800, label=None):
    """Рисует плоскую параметрическую кривую t -> (x(t), y(t))."""
    t = np.linspace(t_range[0], t_range[1], n)
    xy = np.asarray(xy_func(t), dtype=float)
    ax.plot(xy[0], xy[1], label=label)
    if label:
        ax.legend()
    return xy


def plot_implicit_2d(ax, F, xlim, ylim, n=500, level=0, label=None):
    """Рисует неявную кривую F(x,y)=level через contour."""
    xs = np.linspace(xlim[0], xlim[1], n)
    ys = np.linspace(ylim[0], ylim[1], n)
    X, Y = np.meshgrid(xs, ys)
    Z = F(X, Y)
    cs = ax.contour(X, Y, Z, levels=[level])
    if label:
        cs.collections[0].set_label(label)
        ax.legend()
    return cs

In [ ]:
def set_axes_equal_3d(ax):
    """Делает масштабы по осям 3D одинаковыми."""
    x_limits = ax.get_xlim3d()
    y_limits = ax.get_ylim3d()
    z_limits = ax.get_zlim3d()
    x_range = abs(x_limits[1] - x_limits[0])
    y_range = abs(y_limits[1] - y_limits[0])
    z_range = abs(z_limits[1] - z_limits[0])
    radius = 0.5 * max([x_range, y_range, z_range])
    x_middle = np.mean(x_limits)
    y_middle = np.mean(y_limits)
    z_middle = np.mean(z_limits)
    ax.set_xlim3d([x_middle - radius, x_middle + radius])
    ax.set_ylim3d([y_middle - radius, y_middle + radius])
    ax.set_zlim3d([z_middle - radius, z_middle + radius])


def setup_3d(xlim=(-5, 5), ylim=(-5, 5), zlim=(-5, 5), title=None):
    """Создает 3D-систему координат."""
    fig = plt.figure(figsize=(8, 7))
    ax = fig.add_subplot(111, projection='3d')
    ax.set_xlim(*xlim)
    ax.set_ylim(*ylim)
    ax.set_zlim(*zlim)
    ax.set_xlabel('x')
    ax.set_ylabel('y')
    ax.set_zlabel('z')
    if title:
        ax.set_title(title)
    # оси координат
    ax.plot([xlim[0], xlim[1]], [0, 0], [0, 0], linewidth=1)
    ax.plot([0, 0], [ylim[0], ylim[1]], [0, 0], linewidth=1)
    ax.plot([0, 0], [0, 0], [zlim[0], zlim[1]], linewidth=1)
    return fig, ax


def plot_points_3d(ax, points, labels=None):
    labels = labels or [None] * len(points)
    for p, label in zip(points, labels):
        p = np.asarray(p, dtype=float)
        ax.scatter(p[0], p[1], p[2], s=45)
        if label:
            ax.text(p[0], p[1], p[2], '  ' + label)


def plot_parametric_3d(ax, r_func, t_range, n=800, label=None):
    """Рисует пространственную параметрическую кривую t -> (x(t), y(t), z(t))."""
    t = np.linspace(t_range[0], t_range[1], n)
    r = np.asarray(r_func(t), dtype=float)
    ax.plot(r[0], r[1], r[2], label=label)
    if label:
        ax.legend()
    return r


def plot_surface_3d(ax, r_func, u_range, v_range, nu=80, nv=80, alpha=0.45, label=None):
    """Рисует параметрическую поверхность r(u,v). r_func должен возвращать массивы X,Y,Z."""
    u = np.linspace(u_range[0], u_range[1], nu)
    v = np.linspace(v_range[0], v_range[1], nv)
    U, V = np.meshgrid(u, v)
    X, Y, Z = r_func(U, V)
    surf = ax.plot_surface(X, Y, Z, alpha=alpha, linewidth=0, antialiased=True)
    if label:
        surf.set_label(label)
    set_axes_equal_3d(ax)
    return X, Y, Z

In [ ]:
def partial_u(r, u, v, h=1e-5):
    return (np.asarray(r(u + h, v)) - np.asarray(r(u - h, v))) / (2*h)


def partial_v(r, u, v, h=1e-5):
    return (np.asarray(r(u, v + h)) - np.asarray(r(u, v - h))) / (2*h)


def second_uu(r, u, v, h=1e-4):
    return (np.asarray(r(u + h, v)) - 2*np.asarray(r(u, v)) + np.asarray(r(u - h, v))) / (h*h)


def second_uv(r, u, v, h=1e-4):
    return (np.asarray(r(u + h, v + h)) - np.asarray(r(u + h, v - h)) - np.asarray(r(u - h, v + h)) + np.asarray(r(u - h, v - h))) / (4*h*h)


def second_vv(r, u, v, h=1e-4):
    return (np.asarray(r(u, v + h)) - 2*np.asarray(r(u, v)) + np.asarray(r(u, v - h))) / (h*h)


def surface_normal(r, u, v):
    ru = partial_u(r, u, v)
    rv = partial_v(r, u, v)
    return normalize(np.cross(ru, rv))


def first_fundamental_form(r, u, v):
    ru = partial_u(r, u, v)
    rv = partial_v(r, u, v)
    E = float(np.dot(ru, ru))
    F = float(np.dot(ru, rv))
    G = float(np.dot(rv, rv))
    return np.array([[E, F], [F, G]])


def second_fundamental_form(r, u, v):
    m = surface_normal(r, u, v)
    L = float(np.dot(second_uu(r, u, v), m))
    M = float(np.dot(second_uv(r, u, v), m))
    N = float(np.dot(second_vv(r, u, v), m))
    return np.array([[L, M], [M, N]])


def tangent_plane_patch(r, u0, v0, su=1.0, sv=1.0, n=12):
    """Патч касательной плоскости через r(u0,v0)."""
    p = np.asarray(r(u0, v0), dtype=float)
    ru = partial_u(r, u0, v0)
    rv = partial_v(r, u0, v0)
    a = np.linspace(-su, su, n)
    b = np.linspace(-sv, sv, n)
    A, B = np.meshgrid(a, b)
    X = p[0] + A*ru[0] + B*rv[0]
    Y = p[1] + A*ru[1] + B*rv[1]
    Z = p[2] + A*ru[2] + B*rv[2]
    return X, Y, Z


def plot_tangent_plane_and_normal(ax, r, u0, v0, plane_scale=0.5, normal_scale=1.0):
    p = np.asarray(r(u0, v0), dtype=float)
    X, Y, Z = tangent_plane_patch(r, u0, v0, plane_scale, plane_scale)
    ax.plot_surface(X, Y, Z, alpha=0.30, linewidth=0)
    m = surface_normal(r, u0, v0)
    ax.quiver(p[0], p[1], p[2], normal_scale*m[0], normal_scale*m[1], normal_scale*m[2], arrow_length_ratio=0.15)
    plot_points_3d(ax, [p], ['M'])
    set_axes_equal_3d(ax)
    return p, m

## Функции для построения индикатрисы

In [ ]:
def dupin_points_in_tangent_plane(r, u0, v0, n=1200, max_radius=8.0):
    """
    Строит точки индикатрисы Дюпена в 3D как p + alpha*r_u + beta*r_v.
    Использует полярный обход в координатах (alpha,beta).
    """
    p = np.asarray(r(u0, v0), dtype=float)
    ru = partial_u(r, u0, v0)
    rv = partial_v(r, u0, v0)
    B = second_fundamental_form(r, u0, v0)

    theta = np.linspace(0, 2*np.pi, n)
    dirs = np.vstack([np.cos(theta), np.sin(theta)])
    q = np.einsum('ij,ji->i', dirs.T @ B, dirs)
    mask = np.abs(q) > 1e-6
    rho = np.empty_like(theta)
    rho[:] = np.nan
    rho[mask] = 1 / np.sqrt(np.abs(q[mask]))
    rho[rho > max_radius] = np.nan

    alpha = rho * np.cos(theta)
    beta = rho * np.sin(theta)
    X = p[0] + alpha*ru[0] + beta*rv[0]
    Y = p[1] + alpha*ru[1] + beta*rv[1]
    Z = p[2] + alpha*ru[2] + beta*rv[2]
    return alpha, beta, np.vstack([X, Y, Z]), B


def plot_dupin_2d(ax, B, xlim=(-3, 3), ylim=(-3, 3), title=None):
    def F(A, C):
        return np.abs(B[0,0]*A**2 + 2*B[0,1]*A*C + B[1,1]*C**2) - 1
    plot_implicit_2d(ax, F, xlim, ylim, n=600)
    if title:
        ax.set_title(title)

## Выбор случая из 7.18

In [ ]:
R = 2.0
a = 1.5
case = 'a'  # 'a', 'b', 'c', 'd'

if case == 'a':
    # Сфера: r(u,v)=(R cos u cos v, R cos u sin v, R sin u), точка u=v=pi/4.
    def r(u, v):
        return np.array([R*np.cos(u)*np.cos(v), R*np.cos(u)*np.sin(v), R*np.sin(u)], dtype=float)
    u0, v0 = np.pi/4, np.pi/4
    urange, vrange = (-np.pi/2, np.pi/2), (0, 2*np.pi)
    lim3 = (-2.5, 2.5), (-2.5, 2.5), (-2.5, 2.5)

elif case == 'b':
    # Цилиндр: r(u,v)=(a cos v, a sin v, u), произвольная точка.
    def r(u, v):
        return np.array([a*np.cos(v), a*np.sin(v), u], dtype=float)
    u0, v0 = 0.3, np.pi/4
    urange, vrange = (-2, 2), (0, 2*np.pi)
    lim3 = (-2.2, 2.2), (-2.2, 2.2), (-2.2, 2.2)

elif case == 'c':
    # Параболоид: z=2x^2 + 9/2 y^2, точка — начало координат.
    def r(u, v):
        return np.array([u, v, 2*u**2 + 4.5*v**2], dtype=float)
    u0, v0 = 0.0, 0.0
    urange, vrange = (-1.0, 1.0), (-1.0, 1.0)
    lim3 = (-1.5, 1.5), (-1.5, 1.5), (-0.1, 5)

elif case == 'd':
    # Катеноид: r(u,v)=(cosh u cos v, cosh u sin v, u), произвольная точка.
    def r(u, v):
        return np.array([np.cosh(u)*np.cos(v), np.cosh(u)*np.sin(v), u], dtype=float)
    u0, v0 = 0.6, np.pi/4
    urange, vrange = (-1.4, 1.4), (0, 2*np.pi)
    lim3 = (-3, 3), (-3, 3), (-2, 2)
else:
    raise ValueError('unknown case')

G = first_fundamental_form(r, u0, v0)
B = second_fundamental_form(r, u0, v0)
print('G =\n', G)
print('B =\n', B)
print('Уравнение индикатрисы: |L alpha^2 + 2M alpha beta + N beta^2| = 1')
print('L, M, N =', B[0,0], B[0,1], B[1,1])

## 2D-график индикатрисы в координатах $(lpha,eta)$ касательной плоскости

In [ ]:
fig, ax = setup_2d((-4, 4), (-4, 4), title=f'7.18{case}: индикатриса Дюпена в координатах касательной плоскости', xlabel=r'$\alpha$', ylabel=r'$\beta$')
plot_dupin_2d(ax, B, (-4, 4), (-4, 4))
plt.show()

## 3D-график: поверхность, касательная плоскость и индикатриса

In [ ]:
alpha, beta, pts3d, B = dupin_points_in_tangent_plane(r, u0, v0, n=2000, max_radius=6.0)

fig, ax = setup_3d(*lim3, title=f'7.18{case}: индикатриса на касательной плоскости')
plot_surface_3d(ax, r, urange, vrange, alpha=0.22)
plot_tangent_plane_and_normal(ax, r, u0, v0, plane_scale=0.8, normal_scale=0.6)
ax.plot(pts3d[0], pts3d[1], pts3d[2], linewidth=2, label='Dupin')
ax.legend()
set_axes_equal_3d(ax)
plt.show()

## Место для финального аналитического ответа

Для каждого подпункта нужно записать:

1. $r_u,r_v$;
2. единичную нормаль $m$;
3. $L=(r_{uu},m)$, $M=(r_{uv},m)$, $N=(r_{vv},m)$;
4. уравнение $|L\alpha^2+2M\alpha\beta+N\beta^2|=1$;
5. тип индикатрисы: эллипс, гипербола или вырожденный случай.